<a href="https://colab.research.google.com/github/abhishekrs2003/ICT_DSA_2026/blob/main/Dat_Acquisition_Casestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dat Acquisition Casestudy

### Load SpaceX Launch Data from API

In [27]:
import requests
import pandas as pd

url=" https://api.spacexdata.com/v4/launches"
launch_response = requests.get(url)
launch_data = launch_response.json()

launch_df = pd.DataFrame(launch_data)

launch_df = launch_df[['name', 'date_utc', 'success', 'details', 'rocket']]

launch_df['date_utc'] = pd.to_datetime(launch_df['date_utc'])

launch_df['year'] = launch_df['date_utc'].dt.year

print("Launch Data Loaded")
launch_df.head()

Launch Data Loaded


,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009


### Load Rocket Metadata

In [28]:
rocket_url = "https://api.spacexdata.com/v4/rockets"

rocket_response = requests.get(rocket_url)
rocket_data = rocket_response.json()

rocket_df = pd.DataFrame(rocket_data)

rocket_df = rocket_df[['id', 'name', 'type', 'active', 'stages']]

print("Rocket Data Loaded")
rocket_df.head()

Rocket Data Loaded


,id,name,type,active,stages
0,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
2,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2
3,5e9d0d96eda699382d09d1ee,Starship,rocket,False,2


###Merge Launch and Rocket Data

In [29]:
merged_df = pd.merge(launch_df, rocket_df,left_on='rocket',right_on='id',how='left')
merged_df.head()

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2


###Add Simulated Country Information

In [35]:
import random

countries = ['USA', 'Russia', 'India', 'China', 'France']

merged_df['country'] = [random.choice(countries) for _ in range(len(merged_df))]

merged_df[['name_x', 'country']].head()

,name_x,country
0,FalconSat,France
1,DemoSat,China
2,Trailblazer,France
3,RatSat,Russia
4,RazakSat,China


###Store Merged Data in SQLite3

In [39]:
import sqlite3

conn = sqlite3.connect("spacex_launches.db")

merged_df.to_sql('launches',conn, if_exists='replace',index=False)

print("Data Stored in Database")


Data Stored in Database


###Launches by Country

In [31]:
query1 = """
SELECT country,COUNT(*) AS launch_count
FROM launches
GROUP BY country
ORDER BY launch_count DESC;
"""

c_launches = pd.read_sql(query1, conn)

c_launches

,country,launch_count
0,Russia,45
1,USA,44
2,France,41
3,India,39
4,China,36


###Which year had the highest number of launches?

In [32]:
query2 = """
SELECT year,COUNT(*) AS total_launches
FROM launches
GROUP BY year
ORDER BY total_launches DESC
LIMIT 1;
"""

h_year = pd.read_sql(query2, conn)

h_year

,year,total_launches
0,2022,62


###Top 5 Missions by Launch Count

In [33]:
query3 = """
SELECT name_x,COUNT(*) AS launch_count
FROM launches
GROUP BY name_x
ORDER BY launch_count DESC
LIMIT 5;
"""

t_missions = pd.read_sql(query3, conn)

t_missions

,name_x,launch_count
0,ispace Mission 1 & Rashid,1
1,ZUMA,1
2,WorldView Legion 1 & 2,1
3,Viasat-3 & Arcturus,1
4,USSF-44,1
